# EDA Consolidado - Panvel

Este notebook consolida a Análise Exploratória de Dados (EDA) realizada pelo grupo em uma narrativa única. Os notebooks individuais permanecem preservados em `notebooks/00_referencias/`, enquanto este arquivo passa a ser a versão oficial para revisão, apresentação e tomada de decisão da etapa exploratória.


## Objetivos

1. Conferir a qualidade e a consistência das bases.
2. Entender o escopo de filiais utilizado na pipeline.
3. Consolidar os principais achados sobre vendas, metas, devoluções e cadastro de filiais.
4. Registrar decisões para a preparação da base e para a próxima etapa de features.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

def encontrar_raiz_projeto(inicio=None):
    inicio = Path.cwd() if inicio is None else Path(inicio)
    candidatos = [inicio, *inicio.parents]
    pastas_obrigatorias = ["Base_Origi", "Base_V1", "Base_exo"]

    for candidato in candidatos:
        if all((candidato / pasta).exists() for pasta in pastas_obrigatorias):
            return candidato

    raise FileNotFoundError(
        "Nao foi possivel encontrar a raiz do projeto. "
        "Execute o notebook a partir da raiz ou mantenha as pastas Base_Origi, Base_V1 e Base_exo na estrutura esperada."
    )


ROOT = encontrar_raiz_projeto()
BASE_ORIGI = ROOT / "Base_Origi"
BASE_V1 = ROOT / "Base_V1"
BASE_EXO = ROOT / "Base_exo"

ARQUIVOS = {
    "orig_vendas": BASE_ORIGI / "project-puc_vendas.parquet",
    "orig_metas": BASE_ORIGI / "project-puc_metas.parquet",
    "orig_filiais": BASE_ORIGI / "project-puc_filiais.parquet",
    "orig_filiais_dt": BASE_ORIGI / "project-puc_filiais_dt_abertura.parquet",
    "orig_devolucoes": BASE_ORIGI / "project-puc_devolucoes.parquet",
    "v1_vendas": BASE_V1 / "vendas_V1.parquet",
    "v1_metas": BASE_V1 / "metas_V1.parquet",
    "v1_filiais": BASE_V1 / "filiais_V1.parquet",
    "v1_vendas_diaria": BASE_V1 / "vendas_diaria_V1.parquet",
    "exo_vendas": BASE_EXO / "vendas_exo.parquet",
    "exo_metas": BASE_EXO / "metas_exo.parquet",
    "exo_filiais": BASE_EXO / "filiais_exo.parquet",
    "exo_vendas_diaria": BASE_EXO / "vendas_diaria_exo.parquet",
}

def parquet_info(path: Path) -> dict:
    pf = pq.ParquetFile(path)
    return {
        "linhas": pf.metadata.num_rows,
        "colunas": len(pf.schema_arrow.names),
        "tamanho_mb": path.stat().st_size / 1024**2,
    }

def read_cols(nome: str, cols: list[str]) -> pd.DataFrame:
    return pd.read_parquet(ARQUIVOS[nome], columns=cols)


## 1. Inventário das bases

Primeiro, conferimos o tamanho de cada camada usada no projeto, observando quantidade de linhas, colunas e filiais distintas.


In [2]:
inventario = (
    pd.DataFrame.from_dict({nome: parquet_info(path) for nome, path in ARQUIVOS.items()}, orient="index")
    .reset_index(names="base")
)
inventario["linhas"] = inventario["linhas"].astype(int)
inventario["colunas"] = inventario["colunas"].astype(int)
inventario["tamanho_mb"] = inventario["tamanho_mb"].round(2)
inventario

,base,linhas,colunas,tamanho_mb
0,orig_vendas,20863735,6,207.35
1,orig_metas,81268,5,1.60
2,orig_filiais,124,10,0.00
3,orig_filiais_dt,124,2,0.00
4,orig_devolucoes,85677,5,0.66
5,v1_vendas,19565924,7,187.88
6,v1_metas,73099,6,1.19
7,v1_filiais,100,13,0.01
8,v1_vendas_diaria,72069,8,1.61
9,exo_vendas,1191251,7,8.94


## 2. Cobertura de filiais

Nas notas do Bruno. Há uma filial transacional, `1704`, que aparece em vendas, metas e devoluções, mas não aparece no cadastro. Além disso, a pipeline separa 100 filiais principais e 24 filiais na `Base_exo`(Celso).


In [3]:
filiais_por_base = {}

for nome, path in ARQUIVOS.items():
    cols = pq.ParquetFile(path).schema_arrow.names
    if "codigo_filial" in cols:
        s = pd.read_parquet(path, columns=["codigo_filial"])["codigo_filial"].astype(str)
        filiais_por_base[nome] = set(s.unique())

pd.DataFrame(
    [{"base": nome, "filiais_distintas": len(filiais)} for nome, filiais in filiais_por_base.items()]
).sort_values("base")

,base,filiais_distintas
11,exo_filiais,24
10,exo_metas,24
9,exo_vendas,24
12,exo_vendas_diaria,24
4,orig_devolucoes,125
2,orig_filiais,124
3,orig_filiais_dt,124
1,orig_metas,125
0,orig_vendas,125
7,v1_filiais,100


In [4]:
orig = filiais_por_base["orig_filiais"]
v1 = filiais_por_base["v1_filiais"]
exo = filiais_por_base["exo_filiais"]

checks_filiais = {
    "filiais_originais": len(orig),
    "filiais_v1": len(v1),
    "filiais_exo": len(exo),
    "v1_mais_exo": len(v1 | exo),
    "intersecao_v1_exo": len(v1 & exo),
    "orig_menos_v1_exo": sorted(orig - (v1 | exo)),
    "v1_exo_menos_orig": sorted((v1 | exo) - orig),
}

checks_filiais


{'filiais_originais': 124,
 'filiais_v1': 100,
 'filiais_exo': 24,
 'v1_mais_exo': 124,
 'intersecao_v1_exo': 0,
 'orig_menos_v1_exo': [],
 'v1_exo_menos_orig': []}

### Filial 1704

A filial `1704` aparece nas bases de vendas, metas e devoluções, mas não aparece em `filiais` nem em `filiais_dt_abertura`. Por falta de cadastro confiável, a decisão atual é mantê-la fora da modelagem.


In [5]:
linhas_1704 = []
for nome, data_col in [
    ("orig_vendas", "data_emissao"),
    ("orig_metas", "data_meta_venda"),
    ("orig_devolucoes", "data_devolucao"),
    ("orig_filiais", None),
    ("orig_filiais_dt", None),
]:
    cols = ["codigo_filial"] + ([data_col] if data_col else [])
    df = read_cols(nome, cols)
    mask = df["codigo_filial"].astype(str).eq("1704")
    row = {"base": nome, "linhas_1704": int(mask.sum())}
    if data_col and mask.any():
        datas = pd.to_datetime(df.loc[mask, data_col], errors="coerce")
        row.update({"data_min": datas.min(), "data_max": datas.max(), "datas_distintas": datas.dt.date.nunique()})
    linhas_1704.append(row)

pd.DataFrame(linhas_1704)

,base,linhas_1704,data_min,data_max,datas_distintas
0,orig_vendas,106560,2024-01-01 00:00:00+00:00,2025-08-18 00:00:00+00:00,595.0
1,orig_metas,609,2024-01-01 03:00:00+00:00,2025-08-31 03:00:00+00:00,609.0
2,orig_devolucoes,495,2024-01-03 03:00:00+00:00,2025-08-14 03:00:00+00:00,371.0
3,orig_filiais,0,NaT,NaT,NaN
4,orig_filiais_dt,0,NaT,NaT,NaN


## 3. Datas e calendário

As bases principais cobrem os anos de 2024 e 2025. Esta checagem também identifica pequenas ausências de datas, que precisam ser consideradas antes da etapa de modelagem.


In [6]:
date_specs = {
    "orig_vendas": "data_emissao",
    "orig_metas": "data_meta_venda",
    "orig_devolucoes": "data_devolucao",
    "v1_vendas": "data_emissao_data",
    "v1_metas": "data_meta_venda_data",
    "v1_vendas_diaria": "data",
}

linhas_datas = []
for nome, col in date_specs.items():
    s = pd.to_datetime(read_cols(nome, [col])[col], errors="coerce")
    linhas_datas.append({
        "base": nome,
        "coluna": col,
        "data_min": s.min(),
        "data_max": s.max(),
        "nulos": int(s.isna().sum()),
        "datas_distintas": int(s.dt.date.nunique()),
    })

pd.DataFrame(linhas_datas)


,base,coluna,data_min,data_max,nulos,datas_distintas
0,orig_vendas,data_emissao,2024-01-01 00:00:00+00:00,2025-12-31 00:00:00+00:00,0,731
1,orig_metas,data_meta_venda,2024-01-01 03:00:00+00:00,2025-12-31 03:00:00+00:00,0,731
2,orig_devolucoes,data_devolucao,2024-01-01 03:00:00+00:00,2025-12-31 03:00:00+00:00,0,731
3,v1_vendas,data_emissao_data,2024-01-01 00:00:00,2025-12-31 00:00:00,0,731
4,v1_metas,data_meta_venda_data,2024-01-01 00:00:00,2025-12-31 00:00:00,0,731
5,v1_vendas_diaria,data,2024-01-01 00:00:00,2025-12-31 00:00:00,0,731


## 4. Nulos e duplicatas

Este bloco consolida a checagem de qualidade dos dados. Nas bases oficiais avaliadas, não foram encontrados nulos relevantes nem duplicatas nas chaves principais.


In [7]:
bases_nulos = [
    "orig_filiais",
    "orig_filiais_dt",
    "orig_metas",
    "orig_devolucoes",
    "orig_vendas",
    "v1_filiais",
    "v1_metas",
    "v1_vendas_diaria",
    "exo_filiais",
    "exo_metas",
    "exo_vendas_diaria",
]

resumo_nulos = []
for nome in bases_nulos:
    df = pd.read_parquet(ARQUIVOS[nome])
    nulls = df.isna().sum()
    resumo_nulos.append({
        "base": nome,
        "colunas_com_nulos": int((nulls > 0).sum()),
        "total_nulos": int(nulls.sum()),
    })

pd.DataFrame(resumo_nulos)


,base,colunas_com_nulos,total_nulos
0,orig_filiais,0,0
1,orig_filiais_dt,0,0
2,orig_metas,0,0
3,orig_devolucoes,0,0
4,orig_vendas,0,0
5,v1_filiais,0,0
6,v1_metas,0,0
7,v1_vendas_diaria,0,0
8,exo_filiais,0,0
9,exo_metas,0,0


In [8]:
checks_chaves = [
    ("orig_filiais", ["codigo_filial"]),
    ("orig_filiais_dt", ["codigo_filial"]),
    ("orig_metas", ["codigo_filial", "data_meta_venda"]),
    ("v1_filiais", ["codigo_filial"]),
    ("v1_metas", ["codigo_filial", "data_meta_venda_data"]),
    ("v1_vendas_diaria", ["codigo_filial", "data"]),
    ("exo_filiais", ["codigo_filial"]),
    ("exo_metas", ["codigo_filial", "data_meta_venda_data"]),
    ("exo_vendas_diaria", ["codigo_filial", "data"]),
]

duplicatas = []
for nome, keys in checks_chaves:
    df = read_cols(nome, keys)
    duplicatas.append({"base": nome, "chave": " + ".join(keys), "linhas_duplicadas": int(df.duplicated(keys).sum())})

pd.DataFrame(duplicatas)


,base,chave,linhas_duplicadas
0,orig_filiais,codigo_filial,0
1,orig_filiais_dt,codigo_filial,0
2,orig_metas,codigo_filial + data_meta_venda,0
3,v1_filiais,codigo_filial,0
4,v1_metas,codigo_filial + data_meta_venda_data,0
5,v1_vendas_diaria,codigo_filial + data,0
6,exo_filiais,codigo_filial,0
7,exo_metas,codigo_filial + data_meta_venda_data,0
8,exo_vendas_diaria,codigo_filial + data,0


## 5. Vendas e faturamento

Ao consolidar os EDAs do grupo, foi identificado que todos tratam `faturamento` como faturamento bruto. O faturamento líquido só é calculado quando as devoluções são incorporadas. A recomendação atual, portanto, é usar o faturamento bruto como alvo principal.


In [9]:
def resumo_vendas(nome: str) -> dict:
    df = read_cols(nome, ["categoria_gerencial", "faturamento", "quantidade"])
    return {
        "base": nome,
        "linhas": len(df),
        "faturamento_total": df["faturamento"].sum(),
        "quantidade_total": df["quantidade"].sum(),
        "faturamento_negativo": int((df["faturamento"] < 0).sum()),
        "quantidade_negativa": int((df["quantidade"] < 0).sum()),
        "categorias": df["categoria_gerencial"].value_counts(dropna=False).to_dict(),
    }

pd.DataFrame([resumo_vendas("orig_vendas"), resumo_vendas("v1_vendas")])

,base,linhas,faturamento_total,quantidade_total,faturamento_negativo,quantidade_negativa,categorias
0,orig_vendas,20863735,5.945417e+09,153414894.0000,0,0,"{'MED': 10476232, 'N-MED': 10387503}"
1,v1_vendas,19565924,5.620321e+09,143661117,0,0,"{'MED': 9848249, 'N-MED': 9717675}"


## 6. Metas

A análise do Arthur chamou atenção para a qualidade das metas, especialmente para casos zerados. Na `Base_V1`, as metas negativas já foram tratadas, mas ainda existem metas zeradas que precisam ser avaliadas antes da modelagem.


In [10]:
def resumo_metas(nome: str) -> dict:
    df = pd.read_parquet(ARQUIVOS[nome])
    return {
        "base": nome,
        "linhas": len(df),
        "zero_valor_meta": int((df["valor_meta_venda"] == 0).sum()),
        "neg_meta_med": int((df["meta_med"] < 0).sum()),
        "neg_meta_n_med": int((df["meta_n_med"] < 0).sum()),
        "neg_valor_meta": int((df["valor_meta_venda"] < 0).sum()),
    }

pd.DataFrame([resumo_metas("orig_metas"), resumo_metas("v1_metas")])

,base,linhas,zero_valor_meta,neg_meta_med,neg_meta_n_med,neg_valor_meta
0,orig_metas,81268,1491,62,0,0
1,v1_metas,73099,1126,0,0,0


In [11]:
metas_v1 = read_cols("v1_metas", ["codigo_filial", "data_meta_venda_data", "valor_meta_venda", "meta_med", "meta_n_med"])
metas_zero = metas_v1[metas_v1["valor_meta_venda"].eq(0)].copy()

display({
    "linhas_meta_zero": len(metas_zero),
    "filiais_com_meta_zero": metas_zero["codigo_filial"].astype(str).nunique(),
    "data_min": pd.to_datetime(metas_zero["data_meta_venda_data"]).min(),
    "data_max": pd.to_datetime(metas_zero["data_meta_venda_data"]).max(),
})

metas_zero["codigo_filial"].astype(str).value_counts().head(10)

{'linhas_meta_zero': 1126,
 'filiais_com_meta_zero': 25,
 'data_min': Timestamp('2024-01-01 00:00:00'),
 'data_max': Timestamp('2025-12-28 00:00:00')}

codigo_filial
1509    126
1548    125
1500    124
1578    117
1590    112
1749    109
1734     97
1716     81
1599     61
1695     52
Name: count, dtype: int64

## 7. Devoluções

As devoluções são importantes para diagnóstico, mas não devem substituir o faturamento bruto como alvo sem validação da regra de negócio. Em alguns dias, o valor devolvido supera o faturamento bruto, gerando faturamento líquido negativo.


In [12]:
vendas_diaria = read_cols("v1_vendas_diaria", ["codigo_filial", "data", "faturamento_bruto_dia", "valor_devolucao_dia", "faturamento_liquido_dia"])

neg_liquido = vendas_diaria[vendas_diaria["faturamento_liquido_dia"] < 0].copy()
neg_liquido.sort_values("faturamento_liquido_dia")

,codigo_filial,data,faturamento_bruto_dia,valor_devolucao_dia,faturamento_liquido_dia
38422,1671,2024-12-18,106947.06,226205.55,-119258.49
36791,1665,2024-07-02,114873.92,208274.18,-93400.26
61929,1776,2024-04-04,58976.70,127008.13,-68031.43
14791,1563,2025-06-12,79405.83,120516.34,-41110.51
24935,1608,2025-12-29,50560.56,61070.84,-10510.28
8263,1536,2025-04-03,65116.31,66519.72,-1403.41


## 8. Perfil das filiais

Este bloco consolida o perfil das filiais trabalhado pelo grupo, incluindo idade, faixa de vida, localidade, tipo de estabelecimento, metragem e serviços disponíveis na loja.


In [13]:
filiais = pd.read_parquet(ARQUIVOS["orig_filiais"])

for col in ["faixa_vida", "localidade", "tipo_estabelecimento", "delivery", "panvel_clinic", "estacionamento", "atendimento_24_horas"]:
    print(f"\n## {col}")
    display(filiais[col].value_counts(dropna=False).head(12).to_frame("quantidade"))

filiais[["metragem_area_venda"]].describe()


## faixa_vida


,quantidade
faixa_vida,
MAIS DE 3 ANOS,94
ENTRE 1-2 ANOS,12
MENOS DE 1 ANO,11
ENTRE 2-3 ANOS,7



## localidade


,quantidade
localidade,
CURITIBA,53
LONDRINA,14
MARINGA,13
CASCAVEL,4
FOZ DO IGUACU,3
TOLEDO,3
PONTA GROSSA,3
SAO JOSE DOS PINHAIS,3
PARANAVAI,2



## tipo_estabelecimento


...


## estacionamento


,quantidade
estacionamento,
SIM,111
NÃO,13



## atendimento_24_horas


,quantidade
atendimento_24_horas,
NÃO,120
SIM,4


,metragem_area_venda
count,124.000000
mean,510.703274
std,101.204792
min,263.760000
25%,470.552550
50%,511.113500
75%,549.500000
max,1140.730600
